# SituatiONION V6 — Relational Dynamics, Geometry, and Abstraction

## Central Question

**How do control-validated situation relations develop across transformer depth, and is the emergence of relational abstraction associated with systematic changes in representational migration and geometry?**

## Core Experiments

1. **Relational Representational Migration**
2. **Relational Geometric Dynamics**
3. **Migration–Geometry Coupling**
4. **Relation Lens**
5. **Relational Abstraction + Entity Invariance**
6. **Dynamics → Abstraction Integration**

## Model

- **Model:** GPT-2 XL
- **Transformer layers:** 48
- **Hidden size:** 1600

## Relations

- `role_assignment`
- `cause_holder_role`
- `temporal_relation`

## Simple Controls

- `event_state`
- `polarity`

## Operational Definition of Migration

\[
\boxed{
\text{Migration}
=
\text{layer-to-layer redistribution of control-adjusted relational accessibility across readouts}
}
\]

This notebook uses **migration** as a descriptive name for redistribution of decodability/accessibility. It does **not** assume that a literal packet of information physically moves from one token or layer to another, and it does not make a causal claim.

# PART I — Reproduce the V5 Foundation

This section imports the V5 outputs that V6 depends on and reconstructs the same control-adjusted accessibility quantity used throughout the new analyses.

**Do not recompute V5 from scratch if the atlas and cached hidden states already exist.**

## 1. Imports and Configuration

Define the model, tasks, readouts, random seed, cache paths, output paths, and analysis constants here.

Keep all configuration centralized so the notebook is reproducible and easy to rerun.

In [ ]:
# TODO — Imports and global configuration
from pathlib import Path
import json
import math
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy, pearsonr, spearmanr
from scipy.linalg import subspace_angles

from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split

SEED = 42
MODEL_NAME = "gpt2-xl"
N_LAYERS = 48
HIDDEN_SIZE = 1600

RELATIONAL_TASKS = [
    "role_assignment",
    "cause_holder_role",
    "temporal_relation",
]

CONTROL_TASKS = [
    "event_state",
    "polarity",
]

ALL_TASKS = RELATIONAL_TASKS + CONTROL_TASKS

READOUTS = [
    "changed_token",
    "event_token",
    "final_token",
    "agent_recipient",
    "mean_pool",
    "max_pool",
]

ROOT = Path(".")
V5_DIR = ROOT / "v5"
CACHE_DIR = ROOT / "cache_v6"
RESULTS_DIR = ROOT / "results_v6"
FIGURE_DIR = ROOT / "figures_v6"

for p in [CACHE_DIR, RESULTS_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
random.seed(SEED)

print("Configured V6")
print("Tasks:", ALL_TASKS)
print("Readouts:", READOUTS)

## 2. Load V5 Artifacts

Load the four V5 inputs that V6 needs:

- `v5_atlas`
- `v5_hidden_states`
- `v5_stimulus_metadata`
- `v5_split_metadata`

Then verify that the expected tasks, layers, readouts, and hidden dimensionality are present.

If your actual V5 filenames differ, only edit this cell.

In [ ]:
# TODO — Replace paths below with your actual V5 artifact paths.

V5_ATLAS_PATH = V5_DIR / "v5_atlas.parquet"
V5_HIDDEN_PATH = V5_DIR / "v5_hidden_states.npz"
V5_STIMULUS_META_PATH = V5_DIR / "v5_stimulus_metadata.parquet"
V5_SPLIT_META_PATH = V5_DIR / "v5_split_metadata.parquet"

# Example loading logic.
# Adjust this block once you inspect the exact V5 file formats.

# v5_atlas = pd.read_parquet(V5_ATLAS_PATH)
# v5_hidden_states = np.load(V5_HIDDEN_PATH, allow_pickle=True)
# v5_stimulus_metadata = pd.read_parquet(V5_STIMULUS_META_PATH)
# v5_split_metadata = pd.read_parquet(V5_SPLIT_META_PATH)

# TODO — validation checks
# assert v5_atlas["layer"].nunique() == N_LAYERS
# assert set(ALL_TASKS).issubset(set(v5_atlas["task"].unique()))
# assert set(READOUTS).issubset(set(v5_atlas["readout"].unique()))
# assert inferred_hidden_dim == HIDDEN_SIZE

print("TODO: load V5 artifacts and run integrity checks.")

## 3. Reconstruct Conservative Advantage

For every:

\[
	ext{task} 	imes 	ext{layer} 	imes 	ext{readout}
\]

extract the V5 scores:

- linear
- lexical
- random projection
- majority
- shuffled labels

Define the strongest nonchance control as:

\[
C^{*}=\max(C_{	ext{lexical}}, C_{	ext{random projection}})
\]

and conservative advantage as:

\[
a = BA_{	ext{linear}}-C^{*}
\]

**Random projection is a compressed hidden-state control, not a chance baseline.**

Save the resulting table as `v6_accessibility_raw.parquet`.

In [ ]:
# TODO — Adapt column names to the exact V5 atlas schema.

def build_conservative_advantage_table(v5_atlas: pd.DataFrame) -> pd.DataFrame:
    required = {
        "task", "layer", "readout",
        "linear_score", "lexical_score", "random_projection_score"
    }
    missing = required - set(v5_atlas.columns)
    if missing:
        raise ValueError(f"Missing expected V5 columns: {sorted(missing)}")

    df = v5_atlas.copy()

    df["strongest_nonchance_control"] = df[
        ["lexical_score", "random_projection_score"]
    ].max(axis=1)

    df["conservative_advantage"] = (
        df["linear_score"] - df["strongest_nonchance_control"]
    )

    return df

# accessibility_raw = build_conservative_advantage_table(v5_atlas)
# accessibility_raw.to_parquet(RESULTS_DIR / "v6_accessibility_raw.parquet", index=False)

print("TODO: reconstruct and save conservative advantage table.")

# PART II — V6.1 Relational Representational Migration

## Research Question

> **Where is each relation accessible across the model, and how does that distribution change with depth?**

V5 winner-switch counts reduce each layer to one readout. V6 instead uses the **full six-readout control-adjusted accessibility profile**.

## 4. Build the Accessibility Tensor

For relation \(R\) and layer \(l\):

\[
A_{R,l} = [a_1,a_2,\ldots,a_6]
\]

where each \(a_r\) is conservative advantage for one readout.

Effective shape:

\[
	ext{task} 	imes 	ext{layer} 	imes 	ext{readout}
\]

For the five tasks and six readouts, this is approximately:

\[
5 	imes 48 	imes 6
\]

In [ ]:
# TODO — Convert the long-form conservative-advantage table into
# task × layer × readout accessibility profiles.

def build_accessibility_matrix(accessibility_raw: pd.DataFrame, task: str) -> pd.DataFrame:
    sub = accessibility_raw.query("task == @task")
    matrix = (
        sub.pivot_table(
            index="layer",
            columns="readout",
            values="conservative_advantage",
            aggfunc="mean"
        )
        .reindex(index=range(N_LAYERS), columns=READOUTS)
    )
    return matrix

# accessibility_matrices = {
#     task: build_accessibility_matrix(accessibility_raw, task)
#     for task in ALL_TASKS
# }

print("TODO: build accessibility matrices.")

## 5. Positive Accessibility Distribution

For migration only, clip negative conservative advantage to zero:

\[
A^+_{R,l} = \max(A_{R,l},0)
\]

Then normalize positive values:

\[
P_{R,l} = rac{A^+_{R,l}}{\sum_r A^+_{R,l,r}}
\]

If all six values are zero, leave the distribution undefined (`NaN`).

**Do not replace an all-zero row with a uniform distribution.**

In [ ]:
def positive_accessibility_distribution(values):
    values = np.asarray(values, dtype=float)
    positive = np.maximum(values, 0.0)
    total = np.nansum(positive)

    if not np.isfinite(total) or total <= 0:
        return np.full_like(positive, np.nan, dtype=float)

    return positive / total

# Example:
# P = positive_accessibility_distribution(accessibility_matrices["temporal_relation"].iloc[10].values)

## 6. Compute Migration

For each adjacent layer transition:

\[
M_R(l)=JS(P_{R,l},P_{R,l+1})
\]

Compute transitions:

\[
0ightarrow1,\ 1ightarrow2,\ \ldots,\ 46ightarrow47
\]

Output columns:

`task | layer_from | layer_to | JS_migration`

In [ ]:
def js_migration(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)

    if np.isnan(p).any() or np.isnan(q).any():
        return np.nan

    # scipy returns sqrt(JS divergence); square it to obtain JS divergence.
    return float(jensenshannon(p, q, base=2.0) ** 2)

def compute_migration_table(accessibility_matrices):
    rows = []

    for task, mat in accessibility_matrices.items():
        probs = np.vstack([
            positive_accessibility_distribution(mat.loc[layer].values)
            for layer in range(N_LAYERS)
        ])

        for layer in range(N_LAYERS - 1):
            rows.append({
                "task": task,
                "layer_from": layer,
                "layer_to": layer + 1,
                "JS_migration": js_migration(probs[layer], probs[layer + 1]),
            })

    return pd.DataFrame(rows)

# migration_df = compute_migration_table(accessibility_matrices)
# migration_df.to_parquet(RESULTS_DIR / "v6_migration.parquet", index=False)

## 7. Accessibility Entropy

For each valid accessibility distribution:

\[
H_R(l)=-\sum_rP_r\log P_r
\]

Interpretation:

- **low entropy:** accessibility is concentrated in one or a few readouts
- **high entropy:** accessibility is distributed across readouts

In [ ]:
def accessibility_entropy(p):
    p = np.asarray(p, dtype=float)
    if np.isnan(p).any():
        return np.nan
    return float(entropy(p, base=2))

# TODO:
# Build task × layer entropy table and save to RESULTS_DIR.

## 8. Accessibility Persistence

Measure how similar the accessibility distribution remains over longer depth intervals.

At minimum evaluate:

\[
	ext{similarity}(P_l,P_{l+k})
\]

for several values of \(k\), such as \(k=1,2,4,8\).

Use one preregistered primary similarity metric and treat alternatives as robustness checks.

In [ ]:
# TODO — Choose and preregister one primary persistence metric.
# Example option: 1 - Jensen-Shannon divergence.

PERSISTENCE_LAGS = [1, 2, 4, 8]

def persistence_similarity(p, q):
    if np.isnan(p).any() or np.isnan(q).any():
        return np.nan
    return 1.0 - js_migration(p, q)

# TODO:
# Return columns:
# task, layer_from, layer_to, lag, persistence_similarity

## 9. Migration Figures

Generate:

- `migration_role_assignment.png`
- `migration_cause_holder.png`
- `migration_temporal.png`
- `migration_controls.png`
- `accessibility_heatmaps.png`
- `entropy_by_layer.png`

The most important visualization is the continuous accessibility heatmap:

- **x:** layer
- **y:** readout
- **value:** conservative advantage

In [ ]:
# TODO — Publication figure functions.
# Do not hard-code interpretations into the plot.

def plot_accessibility_heatmap(matrix: pd.DataFrame, title: str, save_path: Path):
    fig, ax = plt.subplots(figsize=(14, 4))
    im = ax.imshow(matrix.T.values, aspect="auto", origin="lower")
    ax.set_xlabel("Layer")
    ax.set_ylabel("Readout")
    ax.set_yticks(range(len(matrix.columns)))
    ax.set_yticklabels(matrix.columns)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="Conservative advantage")
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

print("TODO: create migration/entropy/heatmap figures.")

# PART III — V6.2 Relational Geometric Dynamics

## Research Question

> **Does the representation itself remain geometrically stable, or does it transform across depth?**

This section follows the **same relation across layers** instead of only comparing different relation types at one layer.

## 10. Build Geometry Datasets

For each:

\[
	ext{relation} 	imes 	ext{layer} 	imes 	ext{chosen fixed readout}
\]

construct:

- \(X\): hidden states
- \(y\): relation labels

For the primary cross-layer analysis, use a **fixed semantically coherent readout**.

Do not compare `changed_token @ L16` directly against `max_pool @ L38` and treat the difference as pure layer geometry because that confounds **layer** and **readout**.

In [ ]:
# TODO — Implement a single accessor that retrieves example-level hidden states
# for one task, one layer, and one fixed readout.

PRIMARY_GEOMETRY_READOUT = "changed_token"  # TODO: preregister/justify before running

def get_geometry_dataset(task, layer, readout, hidden_states, metadata):
    """
    Return:
        X: np.ndarray [n_examples, hidden_size]
        y: np.ndarray [n_examples]
        meta: pd.DataFrame aligned with X
    """
    raise NotImplementedError("Adapt to your exact hidden-state cache schema.")

## 11. Learn a Relational Subspace

For every relation and layer, estimate:

\[
S_R(l)
\]

using **one preregistered primary method**.

A simple starting point is a linear discriminative subspace such as LDA or a low-rank supervised projection.

Save the learned basis for every layer.

In [ ]:
# TODO — Primary relational subspace estimator.
# For binary tasks, note that ordinary LDA may produce only one discriminant direction.
# If you want k>1 dimensions, use a deliberately defined low-rank supervised method.

GEOMETRY_SUBSPACE_DIM = 1

def fit_relational_subspace(X, y, n_components=GEOMETRY_SUBSPACE_DIM):
    lda = LinearDiscriminantAnalysis(n_components=min(n_components, len(np.unique(y)) - 1))
    lda.fit(X, y)

    # scalings_ columns span the discriminative directions in feature space.
    basis = lda.scalings_[:, :lda._max_components]

    # Orthonormalize basis for principal-angle calculations.
    q, _ = np.linalg.qr(basis)
    return q

# relation_subspaces = {task: {} for task in RELATIONAL_TASKS}
# for task in RELATIONAL_TASKS:
#     for layer in range(N_LAYERS):
#         X, y, meta = get_geometry_dataset(...)
#         relation_subspaces[task][layer] = fit_relational_subspace(X, y)

## 12. Adjacent-Layer Geometric Movement

Compute principal angles:

\[
	heta_1,\ldots,	heta_k
\]

between:

\[
S_R(l)\quad	ext{and}\quad S_R(l+1)
\]

Define the primary geometric movement statistic as a preregistered Grassmann/principal-angle distance:

\[
D_R(l)=d_G(S_R(l),S_R(l+1))
\]

Save:

`task | layer_from | layer_to | geometric_distance`

In [ ]:
def grassmann_distance(U, V):
    angles = subspace_angles(U, V)
    return float(np.linalg.norm(angles))

def compute_adjacent_geometry(relation_subspaces):
    rows = []
    for task, layer_map in relation_subspaces.items():
        for layer in range(N_LAYERS - 1):
            U = layer_map[layer]
            V = layer_map[layer + 1]
            rows.append({
                "task": task,
                "layer_from": layer,
                "layer_to": layer + 1,
                "geometric_distance": grassmann_distance(U, V),
            })
    return pd.DataFrame(rows)

# geometry_df = compute_adjacent_geometry(relation_subspaces)
# geometry_df.to_parquet(RESULTS_DIR / "v6_geometry_adjacent.parquet", index=False)

## 13. Full Cross-Layer Geometry

For each relation, compare every layer against every other layer to obtain a:

\[
48	imes48
\]

similarity or distance matrix.

This can reveal:

- gradual rotation
- stable blocks
- abrupt changes
- convergence
- late reorganization

In [ ]:
def cross_layer_geometry_matrix(layer_map):
    M = np.full((N_LAYERS, N_LAYERS), np.nan, dtype=float)
    for i in range(N_LAYERS):
        for j in range(N_LAYERS):
            M[i, j] = grassmann_distance(layer_map[i], layer_map[j])
    return M

# cross_layer_geometry = {
#     task: cross_layer_geometry_matrix(relation_subspaces[task])
#     for task in RELATIONAL_TASKS
# }

## 14. Robustness Metrics

After the primary geometric metric is fixed and working, add secondary checks such as:

- CKA
- CCA / SVCCA
- cosine similarity between probe directions

These are **robustness analyses**, not metrics to shop between after seeing results.

In [ ]:
# TODO — Add robustness implementations only after the primary geometry pipeline is frozen.

def linear_cka(X, Y):
    X = X - X.mean(axis=0, keepdims=True)
    Y = Y - Y.mean(axis=0, keepdims=True)
    numerator = np.linalg.norm(X.T @ Y, ord="fro") ** 2
    denominator = (
        np.linalg.norm(X.T @ X, ord="fro")
        * np.linalg.norm(Y.T @ Y, ord="fro")
    )
    return float(numerator / denominator) if denominator > 0 else np.nan

## 15. Cross-Relation Geometry

At each layer compare:

\[
S_{	ext{role}}(l)\leftrightarrow S_{	ext{cause}}(l)
\]

\[
S_{	ext{role}}(l)\leftrightarrow S_{	ext{temporal}}(l)
\]

\[
S_{	ext{cause}}(l)\leftrightarrow S_{	ext{temporal}}(l)
\]

This asks whether different relation families remain distinct or become more geometrically similar with depth.

In [ ]:
RELATION_PAIRS = [
    ("role_assignment", "cause_holder_role"),
    ("role_assignment", "temporal_relation"),
    ("cause_holder_role", "temporal_relation"),
]

# TODO — Compute one geometric distance per pair × layer.
# Output columns:
# relation_a, relation_b, layer, geometric_distance

# PART IV — V6.3 Migration–Geometry Coupling

## Research Question

> **When relational accessibility migrates, does the underlying relational geometry transform too?**

This is the first direct integration of the V6.1 and V6.2 trajectories.

## 16. Merge the Two Trajectories

For every relation and adjacent layer transition, combine:

- migration \(M_R(l)\)
- geometric change \(D_R(l)\)
- accessibility entropy
- conservative advantage summaries

In [ ]:
# TODO — Merge migration_df, geometry_df, entropy table, and accessibility summaries.

# coupling_df = (
#     migration_df
#     .merge(geometry_df, on=["task", "layer_from", "layer_to"], how="inner")
#     .merge(...)
# )

print("TODO: build migration–geometry coupling table.")

## 17. Test Coupling

Calculate:

- Pearson correlation
- Spearman correlation
- example-valid bootstrap confidence intervals

between:

\[
M_R(l)
\]

and:

\[
D_R(l)
\]

Run the analysis separately for each relational task and compare the pattern against `event_state` and `polarity`.

In [ ]:
def correlation_summary(df, x_col="JS_migration", y_col="geometric_distance"):
    clean = df[[x_col, y_col]].dropna()
    if len(clean) < 3:
        return {"n": len(clean), "pearson_r": np.nan, "pearson_p": np.nan,
                "spearman_r": np.nan, "spearman_p": np.nan}

    pr, pp = pearsonr(clean[x_col], clean[y_col])
    sr, sp = spearmanr(clean[x_col], clean[y_col])

    return {
        "n": len(clean),
        "pearson_r": pr,
        "pearson_p": pp,
        "spearman_r": sr,
        "spearman_p": sp,
    }

# TODO — Add preregistered bootstrap scheme.

## 18. Transition-Type Visualization

Use the migration–geometry scatterplot to visualize four conceptual regimes:

| Migration | Geometry | Interpretation |
|---|---|---|
| low | low | stable |
| high | low | accessibility redistribution |
| low | high | geometric transformation |
| high | high | representational reorganization |

Do not turn these into hard scientific categories unless thresholds are specified **before** inspecting results.

In [ ]:
# TODO — Plot JS migration vs geometric distance.
# Prefer continuous values.
# If threshold quadrants are displayed, define thresholds in configuration first.

# PART V — Relation Lens

## Research Question

> **Can hidden states be mapped into a common interpretable relational space?**

This section is inspired by entity-lens-style decoding, but targets **relational state** rather than entity identity alone.

## 19. Build Relation Lens Training Data

For every example collect:

- hidden state
- relation type
- relation label
- entity identity
- template
- verb
- layer
- readout

Example:

```text
Alice broke the vase.
relation = cause_holder
label = AGENT
entity = Alice
```

In [ ]:
# TODO — Build a long-form Relation Lens dataset.

RELATION_LENS_COLUMNS = [
    "example_id",
    "relation_type",
    "relation_label",
    "entity_identity",
    "template",
    "verb",
    "layer",
    "readout",
]

# hidden state can be stored separately and referenced by example_id + layer + readout.

## 20. Split by Entity

Entity identity must be held out explicitly.

Example:

**Train entities:** Alice, Bob, Carol, David  
**Test entities:** Priya, Marcus, Sofia, Ethan

Test entities must never appear in training for the entity-invariance evaluation.

In [ ]:
# TODO — Replace with metadata-derived entity groups rather than hard-coded names.

def split_entities(metadata, entity_col="entity_identity", test_size=0.25, seed=SEED):
    entities = np.array(sorted(metadata[entity_col].dropna().unique()))
    train_entities, test_entities = train_test_split(
        entities,
        test_size=test_size,
        random_state=seed
    )
    return set(train_entities), set(test_entities)

# train_entities, test_entities = split_entities(relation_lens_meta)
# assert train_entities.isdisjoint(test_entities)

## 21. Learn a Relational Transformation

Learn:

\[
T_R:\mathbb{R}^{1600}ightarrow\mathbb{R}^{k}
\]

with a low-dimensional relational space such as \(k=8\) or \(k=16\).

The goal is for examples with the **same relational state** to become similar even when they contain different entities.

In [ ]:
RELATION_LENS_DIM = 8

class LinearRelationLens:
    """
    Minimal scaffold for a low-dimensional linear relation lens.

    Replace/extend this once the exact training objective is chosen.
    """
    def __init__(self, dim=RELATION_LENS_DIM):
        self.dim = dim
        self.projection_ = None

    def fit(self, X, y):
        # TODO:
        # Replace with a preregistered supervised low-rank objective.
        # This placeholder uses LDA where possible.
        n_classes = len(np.unique(y))
        n_components = min(self.dim, n_classes - 1)
        lda = LinearDiscriminantAnalysis(n_components=n_components)
        Z = lda.fit_transform(X, y)

        self.projection_ = lda.scalings_[:, :n_components]
        self.classes_ = np.array(sorted(np.unique(y)))

        # Orthonormalized projection
        self.projection_, _ = np.linalg.qr(self.projection_)
        return self

    def transform(self, X):
        if self.projection_ is None:
            raise RuntimeError("Fit the lens first.")
        return X @ self.projection_

## 22. Learn Relational Prototypes

For each relation label:

\[
\mu_y=rac{1}{N_y}\sum_i z_i
\]

where:

\[
z_i=T_R(h_i)
\]

Classify unseen examples by nearest prototype.

Examples:

- `prototype_agent`
- `prototype_recipient`
- `prototype_before`
- `prototype_after`

In [ ]:
def compute_prototypes(Z, y):
    prototypes = {}
    for label in np.unique(y):
        prototypes[label] = Z[y == label].mean(axis=0)
    return prototypes

def nearest_prototype_predict(Z, prototypes):
    labels = list(prototypes.keys())
    P = np.vstack([prototypes[label] for label in labels])

    preds = []
    for z in Z:
        d = np.linalg.norm(P - z[None, :], axis=1)
        preds.append(labels[int(np.argmin(d))])
    return np.asarray(preds)

## 23. Apply a Consistent Lens Across Depth

Do not train 48 unrelated lenses and then interpret their outputs as one trajectory.

Use a **defined shared training scheme** so that cross-layer comparisons are meaningful.

Then obtain:

\[
L_R(l)
\]

where \(L_R(l)\) is Relation Lens recovery at layer \(l\).

In [ ]:
# TODO — Choose ONE explicit cross-layer lens scheme before running.
#
# Candidate schemes:
# A. Train one lens on a preregistered anchor layer and apply it to all layers.
# B. Learn one shared projection jointly from all training layers.
# C. Learn aligned layer-specific projections with a common prototype space.
#
# Do NOT silently switch schemes after seeing results.

RELATION_LENS_SCHEME = "anchor_layer"
RELATION_LENS_ANCHOR_LAYER = 24  # TODO: preregister/justify

print("Relation Lens scheme:", RELATION_LENS_SCHEME)

## 24. Entity Invariance Test

For entity \(e\), relation \(R\), and layer \(l\), compare a relation-conditioned transformation:

\[
\Delta_R(e,l)
\]

across entities.

Plain-language question:

> Does putting Alice into role X alter her representation in a similar direction to putting Bob into role X?

High cross-entity agreement supports **entity-independent relational structure**.

Low agreement supports more **entity-specific encoding**.

In [ ]:
# TODO — Define delta construction using matched sentence pairs.
#
# Example:
# delta(entity, layer) = h(entity in role A, layer) - h(entity in role B, layer)
#
# Then calculate pairwise cosine similarity between deltas for different entities.

def cosine_similarity(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else np.nan

## 25. Relation Lens Figures

Generate:

- `relation_lens_accuracy_by_layer.png`
- `entity_invariance_by_layer.png`
- `prototype_separation_by_layer.png`
- `relation_lens_vs_v5_probe.png`

In [ ]:
# TODO — Create Relation Lens visualizations from saved result tables.

# PART VI — V6.4 Emergence of Relational Abstraction

## Research Question

> **At what layers do relational representations survive increasingly difficult compositional novelty?**

The key change from a single generalization score is to measure:

\[
G_R(l,k)
\]

where:

- \(l\) = layer
- \(k\) = holdout difficulty

## 26. Generate the Expanded Dataset

Each stimulus must contain explicit metadata:

- `entity_1`
- `entity_2`
- `verb`
- `predicate`
- `template`
- `syntax_family`
- `relation_type`
- `relation_label`

Use substantially more examples than the small V5 headline test sets.

In [ ]:
REQUIRED_STIMULUS_COLUMNS = [
    "example_id",
    "text",
    "entity_1",
    "entity_2",
    "verb",
    "predicate",
    "template",
    "syntax_family",
    "relation_type",
    "relation_label",
]

# TODO — Load or generate V6 stimuli into v6_stimuli.
# assert set(REQUIRED_STIMULUS_COLUMNS).issubset(v6_stimuli.columns)

## 27. Construct Deterministic Holdouts

Use increasingly difficult systematic holdouts:

- **K0:** normal random split
- **K1:** unseen entities
- **K2:** unseen entity pairs
- **K3:** unseen predicates / verbs
- **K4:** unseen templates
- **K5:** unseen entity × verb combinations
- **K6:** unseen entity × verb × template combinations

Save all split assignments so they are exactly reproducible.

In [ ]:
HOLDOUT_LEVELS = {
    "K0": "random",
    "K1": "unseen_entities",
    "K2": "unseen_entity_pairs",
    "K3": "unseen_predicates_verbs",
    "K4": "unseen_templates",
    "K5": "unseen_entity_x_verb",
    "K6": "unseen_entity_x_verb_x_template",
}

# TODO — Implement deterministic split constructors for each holdout.
# Return a dataframe with:
# example_id, holdout, split

## 28. Leakage Audit

Programmatically verify each holdout.

Examples:

```python
assert no_test_entity_in_training_for_K1
assert no_test_pair_in_training_for_K2
assert no_test_template_in_training_for_K4
```

If leakage is detected, **halt the notebook** rather than continuing with invalid results.

In [ ]:
def fail_if_overlap(train_values, test_values, label):
    overlap = set(train_values) & set(test_values)
    if overlap:
        raise RuntimeError(
            f"Leakage detected for {label}: {len(overlap)} overlapping values."
        )

# TODO — Add one leakage audit per holdout level.
# Print a compact PASS/FAIL report.

## 29. Run Every Layer

For every:

\[
	ext{relation} 	imes 	ext{layer} 	imes 	ext{readout} 	imes 	ext{holdout}
\]

calculate:

- linear balanced accuracy
- majority baseline
- lexical control
- random projection control
- shuffled-label control
- conservative advantage

This produces:

\[
G_R(l,k)
\]

In [ ]:
# TODO — Reuse the V5 probing logic exactly where possible.
#
# Output columns should include:
# task, layer, readout, holdout,
# n_train, n_test,
# linear_score,
# majority_score,
# lexical_score,
# random_projection_score,
# shuffled_label_score,
# strongest_nonchance_control,
# conservative_advantage

print("TODO: run depth-resolved compositional holdout probes.")

## 30. Build Abstraction Maps

For each relation, create a heatmap with:

- **rows:** holdout difficulty \(k\)
- **columns:** layer \(l\)
- **value:** conservative advantage

These maps show **when** increasingly systematic relational generalization becomes recoverable across depth.

In [ ]:
# TODO — Produce one abstraction heatmap per relation.
# Recommended source table:
# task × layer × holdout × conservative_advantage

# PART VII — V6.5 Dynamics → Abstraction

## Research Question

> **Are migration and geometric transformation associated with where systematic relational abstraction emerges?**

This is the integration stage.

## 31. Build the Master V6 Atlas

Create one master dataframe containing:

- relation
- layer
- readout
- conservative advantage
- migration
- entropy
- geometric change
- Relation Lens score
- entity invariance
- K0 generalization
- …
- K6 generalization

Save as:

`v6_master_atlas.parquet`

In [ ]:
# TODO — Merge all V6 result tables with explicit keys.
#
# Suggested keys:
# task, layer, readout where readout-specific
# task, layer_from, layer_to for transition metrics
#
# Avoid accidental many-to-many merges.

# master_atlas.to_parquet(RESULTS_DIR / "v6_master_atlas.parquet", index=False)

print("TODO: build and save V6 master atlas.")

## 32. Test the Main Relationships

Primary integrations:

\[
M\leftrightarrow D
\]

Migration vs geometry.

\[
D\leftrightarrow G
\]

Geometry vs abstraction.

\[
M\leftrightarrow G
\]

Migration vs abstraction.

\[
L\leftrightarrow G
\]

Relation Lens recovery vs abstraction.

\[
E\leftrightarrow G
\]

Entity invariance vs abstraction.

In [ ]:
# TODO — Define preregistered correlation / regression analyses.
# Keep task-wise analyses separate before any pooled analysis.
# Report effect sizes and uncertainty, not only p-values.

## 33. Lagged Exploratory Analysis

Explore whether geometric change tends to occur **before, during, or after** abstraction emerges.

Examples:

\[
D(l)\leftrightarrow G(l+1)
\]

\[
D(l)\leftrightarrow G(l+2)
\]

This is a **temporally ordered association**, not a causal effect.

Do not write:

> geometry causes abstraction

Prefer:

> geometric change precedes or is associated with later abstraction

In [ ]:
LAG_VALUES = [0, 1, 2, 4]

# TODO — Compute preregistered lagged associations.
# Be explicit about boundary handling and multiple comparisons.

# PART VIII — Automatic Result Summaries

The notebook should automatically extract numerical landmarks, but it should **not automatically make scientific claims**.

## 34. Generate Summary Statistics

Automatically identify:

- largest migration transition
- largest geometric transition
- peak abstraction layer
- first layer surviving K6
- peak Relation Lens layer
- peak entity-invariance layer
- strongest role/cause similarity
- strongest temporal divergence

In [ ]:
def safe_idxmax(df, value_col):
    clean = df.dropna(subset=[value_col])
    if clean.empty:
        return None
    return clean.loc[clean[value_col].idxmax()].to_dict()

summary = {}

# TODO examples:
# summary["temporal_peak_migration"] = safe_idxmax(
#     migration_df.query("task == 'temporal_relation'"),
#     "JS_migration"
# )
#
# summary["cause_first_K6_layer"] = ...

summary

## 35. Automatic Figure Captions

Generate factual captions from figure metadata.

Example:

> **Figure 7.** Control-adjusted temporal-relation accessibility across 48 GPT-2 XL layers. Darker cells indicate greater conservative advantage relative to the strongest nonchance control.

Captions should describe what is plotted, not declare the theoretical conclusion.

In [ ]:
# TODO — Store figure metadata and generate factual captions programmatically.

figure_captions = {}

# PART IX — Statistical Validation

Validation should be matched to the unit of inference for each claim.

## 36. Bootstrap Uncertainty

Bootstrap at the **example level where valid** for:

- migration
- geometry
- generalization
- entity invariance
- coupling statistics

Store:

- estimate
- CI low
- CI high

In [ ]:
N_BOOTSTRAP = 2000

def bootstrap_ci(values, statistic=np.mean, n_boot=N_BOOTSTRAP, seed=SEED, alpha=0.05):
    values = np.asarray(values)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        sample = rng.choice(values, size=len(values), replace=True)
        boots.append(statistic(sample))

    boots = np.asarray(boots)
    estimate = statistic(values)
    lo = np.quantile(boots, alpha / 2)
    hi = np.quantile(boots, 1 - alpha / 2)
    return float(estimate), float(lo), float(hi)

# IMPORTANT:
# This generic helper is NOT automatically valid for every V6 statistic.
# Use example-level resampling matched to the exact estimator whenever possible.

## 37. Permutation / Null Testing

Build hypothesis-specific nulls.

Possible examples:

- shuffle relation labels
- shuffle layer ordering
- shuffle entity-role assignments

Do not use one generic permutation test for every scientific claim.

In [ ]:
N_PERMUTATIONS = 2000

# TODO — Implement separate null generators per hypothesis.
# Each test should record:
# observed_statistic
# null_mean
# null_sd
# p_value
# seed
# n_permutations

# PART X — Final Report Generation

All publication figures and tables should be reproducible from saved result tables without rerunning expensive hidden-state extraction or probe fitting.

## 38. Final Figures

Create one function that regenerates every final figure from cached result tables:

```python
make_all_figures(master_atlas)
```

In [ ]:
def make_all_figures(master_atlas):
    """
    TODO:
    Call all final plotting functions from one place.
    Every figure should save to FIGURE_DIR.
    """
    raise NotImplementedError

# make_all_figures(master_atlas)

## 39. Final Tables

Automatically create:

1. V5 → V6 baseline
2. Migration statistics
3. Geometry statistics
4. Relation Lens results
5. Compositional generalization
6. Integrated relationships

In [ ]:
# TODO — Build publication-ready result dataframes and export CSV/LaTeX/Markdown as needed.

FINAL_TABLE_NAMES = [
    "table1_v5_to_v6_baseline",
    "table2_migration",
    "table3_geometry",
    "table4_relation_lens",
    "table5_compositional_generalization",
    "table6_integrated_relationships",
]

## 40. Notebook Conclusion

Have Python print the final numerical result summary.

Write the scientific interpretation manually after examining:

- effect sizes
- uncertainty
- robustness checks
- control comparisons
- null tests

Do **not** automate conclusions using rules like:

```python
if p < .05:
    print("The model forms abstract relations!")
```

The notebook should compute evidence. The interpretation remains a scientific judgment.

In [ ]:
# TODO — Print compact numeric summary only.
# Example:
#
# print(json.dumps(summary, indent=2, default=str))
#
# Then add a NEW MARKDOWN CELL manually containing your scientific interpretation.

# Recommended Execution Order

Run the notebook in this dependency order:

\[
oxed{
	ext{V5 ingestion}
ightarrow
	ext{migration}
ightarrow
	ext{geometry}
ightarrow
	ext{coupling}
ightarrow
	ext{Relation Lens}
ightarrow
	ext{abstraction}
ightarrow
	ext{integration}
}
\]

Each section should save intermediate results before the next expensive stage begins.

## Core Saved Artifacts

Recommended outputs:

```text
results_v6/
    v6_accessibility_raw.parquet
    v6_migration.parquet
    v6_entropy.parquet
    v6_persistence.parquet
    v6_geometry_adjacent.parquet
    v6_geometry_cross_layer.npz
    v6_cross_relation_geometry.parquet
    v6_migration_geometry_coupling.parquet
    v6_relation_lens.parquet
    v6_entity_invariance.parquet
    v6_generalization.parquet
    v6_master_atlas.parquet

figures_v6/
    accessibility_heatmaps.png
    entropy_by_layer.png
    migration_role_assignment.png
    migration_cause_holder.png
    migration_temporal.png
    migration_controls.png
    cross_layer_geometry_*.png
    migration_geometry_coupling.png
    relation_lens_accuracy_by_layer.png
    entity_invariance_by_layer.png
    prototype_separation_by_layer.png
    abstraction_*.png
```

## Final Guardrails

- Preserve V5 control definitions wherever V6 extends V5.
- Never call random projection a chance baseline.
- Do not interpret accessibility as causal use.
- Do not interpret migration as literal transport.
- Keep the primary geometry metric fixed before viewing final results.
- Use deterministic compositional splits and explicit leakage audits.
- Use larger V6 stimulus sets than the small V5 headline test sets.
- Keep Relation Lens training/evaluation schemes comparable across depth.
- Treat lagged relationships as associations, not causal evidence.
- Save intermediate tables so expensive computation does not need to be repeated.